In [ ]:
# Generate expanded synthetic dataset for better model training
import random

print("🔄 Expanding dataset with synthetic historical data...")

# Load product hierarchy
products_df = pd.read_csv('product_hierarchy.csv')

# Create 2 years of historical data (original has 1 year)
expanded_transactions = []

# Current data date range
current_min_date = pd.to_datetime('2023-01-01')
current_max_date = pd.to_datetime('2023-12-31')

# Expand to 2022-2024
start_date = pd.to_datetime('2022-01-01')
end_date = pd.to_datetime('2024-12-31')

stores = [1, 2, 3]
skus = products_df['sku_id'].values

# Create transactions for each day
current_date = start_date
while current_date <= end_date:
    for store_id in stores:
        for sku_id in skus:
            # Get product info
            product = products_df[products_df['sku_id'] == sku_id].iloc[0]
            
            # Create realistic demand patterns
            day_of_week = current_date.dayofweek
            month = current_date.month
            
            # Base demand
            base_qty = np.random.normal(35, 8)  # Mean 35, std 8
            
            # Weekend boost
            weekend_multiplier = 1.4 if day_of_week >= 5 else 1.0
            
            # Seasonal variation
            if month == 12:  # December surge
                seasonal_multiplier = 1.5
            elif month in [6, 7]:  # Summer peak
                seasonal_multiplier = 1.3
            else:
                seasonal_multiplier = 1.0
            
            # No-sale probability (5%)
            if random.random() < 0.05:
                qty = 0
                price = 0
            else:
                qty = max(1, int(base_qty * weekend_multiplier * seasonal_multiplier * random.uniform(0.8, 1.2)))
                # Price variation ±10%
                price = product['sell_price'] * random.uniform(0.9, 1.1)
            
            if qty > 0:
                expanded_transactions.append({
                    'date': current_date,
                    'store_id': store_id,
                    'sku_id': sku_id,
                    'qty_sold': qty,
                    'transaction_price': price
                })
    
    current_date += timedelta(days=1)

expanded_df = pd.DataFrame(expanded_transactions)
print(f"✅ Generated {len(expanded_df):,} transactions")

# Merge with product info
expanded_df = expanded_df.merge(
    products_df[['sku_id', 'product_name', 'category', 'cost_price', 'sell_price']],
    on='sku_id',
    how='left'
)

# Calculate metrics
expanded_df['revenue'] = expanded_df['qty_sold'] * expanded_df['transaction_price']
expanded_df['margin'] = (expanded_df['transaction_price'] - expanded_df['cost_price']) * expanded_df['qty_sold']

# Save expanded dataset
expanded_df.to_csv('expanded_historical_transactions.csv', index=False)
print(f"💾 Saved expanded dataset to: expanded_historical_transactions.csv")

print(f"\n📊 EXPANDED DATASET STATISTICS:")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"Date range: {expanded_df['date'].min().date()} to {expanded_df['date'].max().date()}")
print(f"Total transactions: {len(expanded_df):,}")
print(f"Stores: {expanded_df['store_id'].nunique()}")
print(f"SKUs: {expanded_df['sku_id'].nunique()}")
print(f"Avg qty per transaction: {expanded_df['qty_sold'].mean():.2f}")
print(f"Total revenue: ${expanded_df['revenue'].sum():,.2f}")
print(f"Date range: {expanded_df['date'].min().date()} to {expanded_df['date'].max().date()}")

## 📊 Generate Expanded Dataset for Better Predictions

Expand the transaction history to improve model accuracy and enable real-time predictions.

In [ ]:
# Calculate inventory alerts dynamically based on forecast period
alerts_list = []

for store_id in features_df['store_id'].unique():
    for sku_id in features_df['sku_id'].unique():
        store_sku = features_df[(features_df['store_id'] == store_id) & (features_df['sku_id'] == sku_id)].copy()
        
        if len(store_sku) == 0:
            continue
        
        # Get product info
        product_info = cleaned_df[cleaned_df['sku_id'] == sku_id]
        if len(product_info) == 0:
            continue
        
        product_name = product_info['product_name'].iloc[0]
        
        # Get current inventory
        inv = inventory_df[(inventory_df['store_id'] == store_id) & (inventory_df['sku_id'] == sku_id)]
        current_stock = inv['stock_on_hand'].values[0] if len(inv) > 0 else 50
        
        # Calculate forecasted demand for the period
        avg_daily_demand = store_sku['predicted_qty'].mean()
        demand_for_period = avg_daily_demand * days_to_forecast
        
        daily_avg = demand_for_period / days_to_forecast if days_to_forecast > 0 else 1
        lead_time = inv['lead_time_days'].values[0] if len(inv) > 0 else 7
        reorder_point = daily_avg * lead_time + daily_avg * 7
        
        stock_after_demand = current_stock - demand_for_period
        excess_stock = max(0, current_stock - (demand_for_period * 1.2))
        overstock_value = excess_stock * 50  # ~$50 avg unit price
        
        # Determine status
        if stock_after_demand < 0:
            status = "CRITICAL REORDER"
            severity = "critical"
        elif current_stock < reorder_point:
            status = "LOW STOCK WARNING"
            severity = "warning"
        elif current_stock > demand_for_period * 2:
            status = "OVERSTOCK WATCH"
            severity = "warning"
        elif current_stock >= demand_for_period * 0.8 and current_stock <= demand_for_period * 1.2:
            status = "BALANCED - NORMAL"
            severity = "normal"
        else:
            status = "NORMAL"
            severity = "normal"
        
        alerts_list.append({
            'Store': store_id,
            'SKU': sku_id,
            'Product': product_name,
            'Current Stock': int(current_stock),
            f'Forecasted Demand ({days_to_forecast}d)': int(demand_for_period),
            'Stock After Period': int(stock_after_demand),
            'Overstock Value': f"${overstock_value:,.0f}",
            'Status': status,
            'Severity': severity
        })

alerts_df = pd.DataFrame(alerts_list)

# Summary
print(f"\n🚨 INVENTORY ALERTS (for {days_to_forecast}-day period):")
print("━" * 100)
critical = len(alerts_df[alerts_df['Severity'] == 'critical'])
warning = len(alerts_df[alerts_df['Severity'] == 'warning'])
normal = len(alerts_df[alerts_df['Severity'] == 'normal'])
total_overstock_value = alerts_df[alerts_df['Status'] == 'OVERSTOCK WATCH']['Overstock Value'].apply(lambda x: float(x.replace('$', '').replace(',', ''))).sum()

print(f"🔴 CRITICAL REORDER:    {critical} SKUs")
print(f"🟡 WARNINGS:            {warning} SKUs")
print(f"🟢 NORMAL:              {normal} SKUs")
print(f"📦 Total Overstock Risk: ${total_overstock_value:,.0f}")
print("━" * 100)

# Display alerts table
print(f"\n📋 DETAILED ALERTS:")
print(alerts_df.to_string(index=False))

## 🚨 Inventory Alert System

Dynamic alerts that adjust based on the forecast period.

In [ ]:
# Calculate dynamic summary statistics based on forecast period
total_records = len(features_df)
avg_daily_qty = features_df['qty_sold'].mean()
total_predicted_demand = features_df['predicted_qty'].sum()
avg_predicted_per_day = total_predicted_demand / total_records if total_records > 0 else 0

# Scale to forecast period
forecasted_demand_total = avg_predicted_per_day * days_to_forecast
forecasted_demand_per_sku = forecasted_demand_total / features_df['sku_id'].nunique()

print(f"""
📈 SUMMARY STATISTICS (for {days_to_forecast}-day forecast):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total SKUs:                    {features_df['sku_id'].nunique()}
Stores:                        {features_df['store_id'].nunique()}
Historical avg daily qty:      {avg_daily_qty:.2f} units
Predicted avg daily qty:       {avg_predicted_per_day:.2f} units
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total forecasted demand ({days_to_forecast}d):  {forecasted_demand_total:,.0f} units
Avg per SKU ({days_to_forecast}d):          {forecasted_demand_per_sku:,.0f} units
Daily variance (σ):            {features_df['qty_sold'].std():.2f} units
""")

## 📊 Dynamic Summary Statistics

Summary statistics will update based on the forecast period defined above.

In [ ]:
# CONFIGURABLE PARAMETER - Change this to see how forecasts adjust
days_to_forecast = 30  # Change this value (7, 14, 30, 60, 90) to see real-time updates

print(f"📅 Forecast Period: {days_to_forecast} days")
print(f"⏰ Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Retail Demand Forecasting - Interactive Analysis

This notebook provides interactive analysis and forecasting for retail inventory management with configurable forecast horizons.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import pickle
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## ⚙️ Configurable Parameters

Adjust these parameters to see how forecasts and alerts change in real-time:

In [2]:
# Load Data Files
try:
    cleaned_df = pd.read_csv('cleaned_data.csv')
    cleaned_df['date'] = pd.to_datetime(cleaned_df['date'])
    
    features_df = pd.read_csv('features_engineered_data.csv')
    features_df['date'] = pd.to_datetime(features_df['date'])
    
    inventory_df = pd.read_csv('current_inventory.csv')
    
    print("✅ Data loaded successfully!")
    print(f"\nCleaned Data: {cleaned_df.shape}")
    print(f"Features Data: {features_df.shape}")
    print(f"Inventory Data: {inventory_df.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")

# Display sample data
print("\n📊 Cleaned Data Sample:")
print(cleaned_df.head())

print("\n📊 Inventory Data:")
print(inventory_df)

✅ Data loaded successfully!

Cleaned Data: (4124, 12)
Features Data: (4380, 33)
Inventory Data: (15, 4)

📊 Cleaned Data Sample:
        date  store_id  sku_id  qty_sold  transaction_price  \
0 2023-01-01         1    5502        17                 60   
1 2023-01-01         3    2205        13                 85   
2 2023-01-01         3    8802        58                 20   
3 2023-01-01         3    8801        77                 95   
4 2023-01-01         2    2205        11                 85   

        product_name     category  cost_price  sell_price  revenue  margin  \
0  Bluetooth Speaker  Electronics          25          60     1020     595   
1       Coffee Maker         Home          45          85     1105     520   
2     Cotton T-Shirt      Apparel           8          20     1160     696   
3      Running Shoes      Apparel          40          95     7315    4235   
4       Coffee Maker         Home          45          85      935     440   

   margin_pct  
0      1

## 📊 Exploratory Data Analysis (EDA)

Let's explore the cleaned sales data to understand its characteristics.

In [3]:
# Descriptive Statistics
print("\n📝 Descriptive Statistics for Revenue Data:")
print(cleaned_df.describe())

# Revenue Distribution
plt.figure(figsize=(10, 6))
sns.histplot(cleaned_df['revenue'], bins=50, kde=True)
plt.title('Distribution of Revenue')
plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.show()

# Revenue Trend Over Time
plt.figure(figsize=(15, 7))
daily_revenue = cleaned_df.groupby('date')['revenue'].sum()
daily_revenue.plot(kind='line', title='Total Revenue Over Time')
plt.xlabel('Date')
plt.ylabel('Total Revenue')
plt.show()


📝 Descriptive Statistics for Sales Data:
                                date     store_id       sku_id     qty_sold  \
count                           4124  4124.000000  4124.000000  4124.000000   
mean   2023-07-01 01:46:50.863239680     1.999030  6318.703443    33.889913   
min              2023-01-01 00:00:00     1.000000  2205.000000     3.000000   
25%              2023-04-02 00:00:00     1.000000  2205.000000    11.000000   
50%              2023-07-01 00:00:00     2.000000  5502.000000    23.000000   
75%              2023-09-30 00:00:00     3.000000  8801.250000    54.000000   
max              2023-12-31 00:00:00     3.000000  8802.000000   110.000000   
std                              NaN     0.815505  2738.308977    25.493262   

       transaction_price   cost_price   sell_price       revenue       margin  \
count        4124.000000  4124.000000  4124.000000   4124.000000  4124.000000   
mean           64.993938    29.510912    64.993938   2028.626334  1155.283705   
min

KeyError: 'sales'

<Figure size 1000x600 with 0 Axes>

## 🤖 Model Loading and Forecasting

Now, let's load the pre-trained XGBoost model to generate demand forecasts.

In [ ]:
# Load the trained model and scaler
try:
    with open('xgboost_model.pkl', 'rb') as f:
        model_data = pickle.load(f)
    
    model = model_data['model']
    scaler = model_data['scaler']
    feature_cols = model_data['feature_cols']
    
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")

# Prepare features for prediction
X_pred = features_df[feature_cols].astype(float)
X_pred_scaled = scaler.transform(X_pred)

# Generate predictions
features_df['predicted_qty'] = model.predict(X_pred_scaled)

# Display predictions
print("\n🔮 Forecasted Quantities:")
print(features_df[['date', 'store_id', 'sku_id', 'qty_sold', 'predicted_qty']].head())

## 📈 Forecast Visualization

Let's visualize the model's forecasts against the actual sales data to evaluate its performance.

In [13]:
# Select a specific product to visualize
sample_store_id = features_df['store_id'].iloc[0]
sample_sku_id = features_df['sku_id'].iloc[0]

product_df = features_df[(features_df['store_id'] == sample_store_id) & (features_df['sku_id'] == sample_sku_id)].copy()

# Ensure predicted_qty exists, if not regenerate it
if 'predicted_qty' not in features_df.columns:
    X_pred = features_df[feature_cols].copy()
    # Convert boolean columns to int
    bool_cols = X_pred.select_dtypes(include=['bool']).columns
    X_pred[bool_cols] = X_pred[bool_cols].astype(int)
    X_pred = X_pred.astype(float)
    X_pred_scaled = scaler.transform(X_pred)
    features_df['predicted_qty'] = model.predict(X_pred_scaled)

product_df = product_df.copy()
product_df['predicted_qty'] = features_df[(features_df['store_id'] == sample_store_id) & (features_df['sku_id'] == sample_sku_id)]['predicted_qty'].values

# Create the plot
fig = go.Figure()

# Add actual sales data
fig.add_trace(go.Scatter(x=product_df['date'], y=product_df['qty_sold'], mode='lines', name='Actual Sales'))

# Add predicted sales data
fig.add_trace(go.Scatter(x=product_df['date'], y=product_df['predicted_qty'], mode='lines', name='Forecasted Sales', line=dict(dash='dash')))

# Update layout
fig.update_layout(
    title=f'Sales Forecast vs. Actuals for SKU {sample_sku_id} at Store {sample_store_id}',
    xaxis_title='Date',
    yaxis_title='Quantity Sold',
    legend_title='Legend'
)

fig.show()

ValueError: could not convert string to float: 'Home'